In [51]:
%load_ext IPython.extensions.autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

The IPython.extensions.autoreload extension is already loaded. To reload it, use:
  %reload_ext IPython.extensions.autoreload


In [80]:
import sys
sys.path.append('../')
from model import FinData
from model import train_valid_test_split
from model import CatboostFinModel, SVMFinModel
from model import fbeta_metric_long, precision_long, recall_long, fbeta_metric_short, precision_short, recall_short

import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import pandas as pd
import numpy as np

import json
import datetime as dt
import pandas as pd
import optuna
import xgboost as xgb
# import lightgbm as lgb

In [81]:
# Choose your fighter

company_name = 'Gazprom' # Gazprom, Bashneft, MMK, Whoosh, Sber
target_type = 'long' # short
target = 'direction_binary_0' # 'direction_binary_1'
metric_for_optuna = fbeta_metric_long # fbeta_metric_short

In [82]:
# args1 = {
#     "n_estimators": 10000,
#     "depth": 5,
#     "learning_rate": 0.02,
#     "use_label_encoder": False,
#     "reg_lambda": 0.0015,
#     "objective": 'binary:logistic',
#     "eval_metric": "logloss",
#     "tree_method": 'hist',
#     "random_state": 42,
#     "verbose": 0,
#     "early_stopping_rounds": 500
# }

args1 = {
    "iterations": 10000,
    "depth": 5,
    "learning_rate": 0.02,
    "use_best_model": True,
    "l2_leaf_reg": 200,
    "loss_function": 'Logloss',
    "eval_metric": 'Logloss',
    "random_state": 42,
    "verbose": 0,
    "early_stopping_rounds": 500
}

# args1_lgbt = {
#     "n_estimators": 10000,
#     "max_depth": 5,
#     "learning_rate": 0.02,
#     "reg_lambda": 200,
#     "objective": "binary",
#     "metric": "binary_logloss",
#     "random_state": 42,
#     "verbose": -1,
#     "early_stopping_rounds": 500
# }

Вот тут генерим вес (но не свой, а класса) - желательно запускать так, чтобы явно каждому объекту задавать вес, а не как мы раньше через class_weights CatBoost - там внутри еще происходила ребалансировка небольшая из-за того что мы не явно задавали - сейчас прям задаем через sample_weights - И для тренировочной и для ВАЛИДАЦИОННОЙ выборки.

In [83]:
df_path = '../datasets/' + company_name + '_1_min.csv'

start_time = dt.datetime(2024, 2, 1)
end_time = dt.datetime(2024, 4, 30)
cutoff_time = start_time - dt.timedelta(days=90)

findata = FinData(df_path)
findata.restrict_time_down(cutoff_time)
findata.restrict_time_up(end_time + dt.timedelta(days=2))
findata.insert_all()

cat_feats = findata.cat_features
num_feats = findata.numeric_features

data = findata.df


def objective(trial):
    weight_1 = trial.suggest_float("unimp_weight", 0.1, 0.4)
    curr_time = start_time

    fbeta_scores = []

    while curr_time < end_time:
        data = findata.df
        train_df = data[(data.utc >= curr_time - dt.timedelta(days=35)) & (data.utc <= curr_time - dt.timedelta(days=5))]
        val_df = data[(data.utc > curr_time - dt.timedelta(days=5)) & (data.utc <= curr_time)]
        test_df = data[(data.utc > curr_time) & (data.utc <= curr_time + dt.timedelta(days=5))]
        train_sd, val_sd, test_sd = train_df["utc"].iloc[0], val_df["utc"].iloc[0], test_df["utc"].iloc[0]
        train_ed, val_ed, test_ed = train_df["utc"].iloc[-1], val_df["utc"].iloc[-1], test_df["utc"].iloc[-1]
        # print(f"Начало тренировочного периода: {train_sd}. Конец тренировочного периода: {train_ed} \n \
        #             Начало валидационного периода: {val_sd}. Конец валидационного периода: {val_ed} \n \
        #             Начало тестового периода: {test_sd}. Конец тестового периода: {test_ed} \n ")
        X_train, y_train = train_df[cat_feats + num_feats], train_df[target]
        X_val, y_val = val_df[cat_feats + num_feats], val_df[target]
        X_test, y_test = test_df[cat_feats + num_feats], test_df[target]
        close = test_df['close']

        sample_weights_train = pd.Series(1.0, index=train_df.index)
        sample_weights_val = pd.Series(1.0, index=val_df.index)  
        
        mask_train = train_df['close'].shift(-1) > train_df['close'] # TODO - if short, then choose  train_df['close'].shift(-1) < train_df['close']
        mask_val = val_df['close'].shift(-1) > val_df['close']  # TODO - if short, then choose  val_df['close'].shift(-1) < val_df['close']

        sample_weights_train.loc[mask_train] = weight_1
        sample_weights_val.loc[mask_val] = weight_1  

        model = CatboostFinModel(args1)  # TODO - put your model here :)
        model.set_datasets(X_train, X_val, y_train, y_val)
        model.set_features(num_feats, cat_feats)
        model.fit(sample_weights_train=sample_weights_train, sample_weights_val=sample_weights_val)

        # model = xgb.XGBClassifier(**args1)
        # model.fit(
        #     X_train, y_train,
        #     sample_weight=sample_weights_train,
        #     eval_set=[(X_val, y_val)],
        #     sample_weight_eval_set = [sample_weights_val]
        #     # eval_sample_weight=[sample_weights_val],
        # )

        # model = lgb.LGBMClassifier(**args1_lgbt)
        # model.fit(
        #     X_train, y_train,
        #     sample_weight=sample_weights_train,
        #     eval_set=[(X_val, y_val)],
        #     eval_sample_weight=[sample_weights_val],
        #     verbose=100,
        #     eval_metric='logloss'
        # )

        proba = model.predict_proba(X_test)[:, 1]
        preds = (proba > 0.5).astype(int)
        fbeta_scores.append(metric_for_optuna(preds, close, commission=0.0008, beta=0.0005))

        curr_time = curr_time + dt.timedelta(days=5)

    return np.median(fbeta_scores)
    

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=7)

print("Лучшие параметры:")
print(study.best_params)
print("Лучшее значение F-beta:")
print(study.best_value)

[I 2025-07-24 17:33:29,154] A new study created in memory with name: no-name-6d0860d6-b37e-447f-93eb-018b5d9aed95
[I 2025-07-24 17:37:22,472] Trial 0 finished with value: 0.5238174884620417 and parameters: {'unimp_weight': 0.32960790278437657}. Best is trial 0 with value: 0.5238174884620417.
[I 2025-07-24 17:41:28,440] Trial 1 finished with value: 0.27120142150017534 and parameters: {'unimp_weight': 0.11571649877494143}. Best is trial 0 with value: 0.5238174884620417.
[I 2025-07-24 17:45:17,652] Trial 2 finished with value: 0.49843979009257433 and parameters: {'unimp_weight': 0.16706162927772422}. Best is trial 0 with value: 0.5238174884620417.
[I 2025-07-24 17:48:59,994] Trial 3 finished with value: 0.4247678756599098 and parameters: {'unimp_weight': 0.15401862568489894}. Best is trial 0 with value: 0.5238174884620417.
[I 2025-07-24 17:52:54,997] Trial 4 finished with value: 0.21045487368419666 and parameters: {'unimp_weight': 0.13376707034824317}. Best is trial 0 with value: 0.523817

Лучшие параметры:
{'unimp_weight': 0.32960790278437657}
Лучшее значение F-beta:
0.5238174884620417


А теперь генерим датасет из полученного параметра =)

In [84]:
import pandas as pd
import datetime as dt

weight_1 = study.best_params['unimp_weight']


cutoff_time = dt.datetime(2024, 1, 1)
start_time = dt.datetime(2024, 2, 1)
end_time = dt.datetime(2024, 4, 30)

res = pd.DataFrame()
findata = FinData(df_path)
findata.restrict_time_down(cutoff_time)
findata.insert_all()

cat_feats = findata.cat_features
num_feats = findata.numeric_features

curr_time = start_time

while curr_time < end_time:
    data = findata.df
    train_df = data[(data.utc >= curr_time - dt.timedelta(days=35)) & (data.utc <= curr_time - dt.timedelta(days=5))]
    val_df = data[(data.utc > curr_time - dt.timedelta(days=5)) & (data.utc <= curr_time)]
    test_df = data[(data.utc > curr_time) & (data.utc <= curr_time + dt.timedelta(days=5))]
    X_train, y_train = train_df[cat_feats + num_feats], train_df[target]
    X_val, y_val = val_df[cat_feats + num_feats], val_df[target]
    X_test, y_test = test_df[cat_feats + num_feats], test_df[target]
    close = test_df['close']

    sample_weights_train = pd.Series(1.0, index=train_df.index)
    sample_weights_val = pd.Series(1.0, index=val_df.index)  
    mask_train = train_df['close'].shift(-1) < train_df['close']
    mask_val = val_df['close'].shift(-1) < val_df['close'] 
    sample_weights_train.loc[mask_train] = weight_1
    sample_weights_val.loc[mask_val] = weight_1  

    # model = xgb.XGBClassifier(**args1)
    # model.fit(
    #     X_train, y_train,
    #     sample_weight=sample_weights_train,
    #     eval_set=[(X_val, y_val)],
    #     sample_weight_eval_set = [sample_weights_val]
    #     # eval_sample_weight=[sample_weights_val],
    # )

    model = CatboostFinModel(args1)
    model.set_datasets(X_train, X_val, y_train, y_val)
    model.set_features(num_feats, cat_feats)
    model.fit(sample_weights_train=sample_weights_train, sample_weights_val=sample_weights_val)

    # model = lgb.LGBMClassifier(**args1_lgbt)
    # model.fit(
    #     X_train, y_train,
    #     sample_weight=sample_weights_train,
    #     eval_set=[(X_val, y_val)],
    #     eval_sample_weight=[sample_weights_val],
    #     eval_metric='logloss'
    # )

    proba = model.predict_proba(X_test)[:, 1]
    temp_df = pd.DataFrame({
        'utc': test_df['utc'].values,
        'open': test_df['open'].values,
        'close': test_df['close'].values,
        'predicted_proba': proba
    })
    res = pd.concat([res, temp_df], ignore_index=True)
    curr_time += dt.timedelta(days=5)
    print(curr_time) # TODO - you can delete this line, if you are tired of tracking time :)
res.to_csv(f'../generated_datasets/xgboost_with_weights/{company_name}_{target_type}-try.csv', index=False)



2024-02-06 00:00:00
2024-02-11 00:00:00
2024-02-16 00:00:00
2024-02-21 00:00:00
2024-02-26 00:00:00
2024-03-02 00:00:00
2024-03-07 00:00:00
2024-03-12 00:00:00
2024-03-17 00:00:00
2024-03-22 00:00:00
2024-03-27 00:00:00
2024-04-01 00:00:00
2024-04-06 00:00:00
2024-04-11 00:00:00
2024-04-16 00:00:00
2024-04-21 00:00:00
2024-04-26 00:00:00
2024-05-01 00:00:00
